# torchmetrics

## The problem without the lib

Computing accuracy per batch and averaging those values is **mathematically wrong** when batches have different sizes:

```python
# ❌ Wrong — average of averages
accs = [(preds == targets).float().mean() for batch in dataloader]
print(sum(accs) / len(accs))

# ✅ Correct — accumulate and divide at the end
total_correct += (preds == targets).sum()
total_samples += len(targets)
print(total_correct / total_samples)
```

`torchmetrics` encapsulates this pattern for you.

---

## Two modes of use

### 1. Compute on the spot (per batch)

Call the metric as a function — does `update + compute` together and returns immediately:

```python
acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)

result = acc(preds, targets)
print(result)  # tensor(0.85)
```

### 2. Accumulate (per epoch)

Accumulates batch by batch, computes at the end, resets for the next epoch:

```python
acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)

for batch in dataloader:
    acc.update(preds, targets)  # accumulates internally

print(acc.compute())  # computes over all batches
acc.reset()           # clears for the next epoch
```

---

## What each method does

| Method | What it does |
|---|---|
| `.update(preds, targets)` | Accumulates internal counters (returns nothing) |
| `.compute()` | Computes the metric over everything accumulated |
| `.reset()` | Clears the accumulators |
| `acc(preds, targets)` | `update` + `compute` together — returns immediately |

---

## When to use each

| Situation | Mode |
|---|---|
| Quick debug, single batch | On the spot — `acc(preds, targets)` |
| Real epoch metric | Accumulate — `update` → `compute` → `reset` |
| PyTorch Lightning | `.log()` with the metric — it manages everything |

---

## MetricCollection — multiple metrics at once

```python
metrics = torchmetrics.MetricCollection([
    torchmetrics.Accuracy(task="multiclass", num_classes=10),
    torchmetrics.F1Score(task="multiclass", num_classes=10),
])

metrics.update(preds, targets)
print(metrics.compute())  # {"Accuracy": ..., "F1Score": ...}
metrics.reset()
```

---

## Note: `task=` is required in versions >= 0.10

```python
Accuracy(task="binary")
Accuracy(task="multiclass", num_classes=N)
Accuracy(task="multilabel", num_labels=N)
```

In [13]:
import torchmetrics
import torch

input = torch.randint(0, 2, (10, 1))
target = torch.randint(0, 2, (10, 1))

acc = torchmetrics.Accuracy(task='binary')
loss = acc(input, target)
print(F'Loss after 1 iteration: {loss:.4f}')

Loss after 1 iteration: 0.4000


In [25]:
import torchmetrics
import torch

iterations = 20
acc = torchmetrics.Accuracy(task='binary')

for x in range(iterations):
    input = torch.randint(0, 2, (10, 1))
    target = torch.randint(0, 2, (10, 1))

    loss = acc(input, target)
    print(f'Loss in the iteration {x}: {loss:.4f}')
    acc.update(input, target)

Result = acc.compute()
print(F'Result after 20 iterations (1 batch): {Result:.4f}')

Loss in the iteration 0: 0.6000
Loss in the iteration 1: 0.7000
Loss in the iteration 2: 0.4000
Loss in the iteration 3: 0.5000
Loss in the iteration 4: 0.6000
Loss in the iteration 5: 0.5000
Loss in the iteration 6: 0.7000
Loss in the iteration 7: 0.5000
Loss in the iteration 8: 0.4000
Loss in the iteration 9: 0.4000
Loss in the iteration 10: 0.2000
Loss in the iteration 11: 0.5000
Loss in the iteration 12: 0.4000
Loss in the iteration 13: 0.8000
Loss in the iteration 14: 0.4000
Loss in the iteration 15: 0.3000
Loss in the iteration 16: 0.4000
Loss in the iteration 17: 0.2000
Loss in the iteration 18: 0.8000
Loss in the iteration 19: 0.6000
Result after 20 iterations (1 batch): 0.4950


In [47]:
import torchmetrics
import torch

iterations = 20
acc = torchmetrics.MetricCollection([
    torchmetrics.Accuracy(task='binary'),
    torchmetrics.F1Score(task='binary')]
)


for x in range(iterations):
    input = torch.randint(0, 2, (10, 1))
    target = torch.randint(0, 2, (10, 1))

    loss = acc(input, target)
    if x % 5 == 0:
        print(f'Loss of Accuracy in the iteration {x}: {loss['BinaryAccuracy']:.4f}')
        print(f'Loss of F1Score in the iteration {x}: {loss['BinaryF1Score']:.4f}')

    acc.update(input, target)

result = acc.compute()
print(F'Result loss Accuracy after 20 iterations (1 batch): {result['BinaryAccuracy']:.4f}')
print(F'Result loss F1Score after 20 iterations (1 batch): {result['BinaryF1Score']:.4f}')

Loss of Accuracy in the iteration 0: 0.6000
Loss of F1Score in the iteration 0: 0.3333
Loss of Accuracy in the iteration 5: 0.4000
Loss of F1Score in the iteration 5: 0.5000
Loss of Accuracy in the iteration 10: 0.6000
Loss of F1Score in the iteration 10: 0.6000
Loss of Accuracy in the iteration 15: 0.5000
Loss of F1Score in the iteration 15: 0.2857
Result loss Accuracy after 20 iterations (1 batch): 0.5150
Result loss F1Score after 20 iterations (1 batch): 0.5446


In [48]:
print(f'Before reset: {acc.compute()["BinaryAccuracy"]:.4f}') 
acc.reset()
print(f'After reset: {acc.compute()["BinaryAccuracy"]:.4f}')   

Before reset: 0.5150
After reset: 0.0000
